# GwenLand glcuda - T4 Ceiling Wave 9 (Direct Model Fetch)

Decision-grade Kaggle A/B: retained Wave 4 versus an opt-in L2-grouped CTA raster for the exact same sm_75 INT8 MMA kernel. Target: **15,000+ prefill tok/s** on one Tesla T4.

External basis: OpenAI Triton's [matrix multiplication tutorial](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html) groups program IDs to improve L2 reuse; AMD Composable Kernel maps block IDs to C tiles through an explicit tile map. Wave 9 tests that scheduling idea without changing MMA operands, dequantization, barriers, or shared-memory layout.


## 1 - Configuration, clean patch stack, and T4 gate

Both arms start at the pinned revision and apply only archived Wave 3 plus retained Wave 4. The candidate then applies the Wave 9-only patch. Rejected/HOLD Waves 5-8 are excluded.


In [ ]:
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import time

REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE9_PATCH_SHA256 = "d569f0967b6cb4c11479eaeb22e20ebfb6fcfa1421ab6c35c51a2160401a5df3"
WAVE9_PATCH_GZIP_B64 = "H4sIAAAAAAACCu1b63bbRpL+r6foaI41oAVCJEhRvESJJdub9YmdZCzNTvZotTAINEmMQAACQJGM5XPmIeYJ90m2qroBNC6klMz8mLMb/pBAdHd1d12/qm663mzG2u25lzL7ZO47K9c+SWLn5I7HAfcT+cpKlmenRpRu2PQZnQ68wOUbNnSn7sh1DYP3R3Z3NmTdTmfQ7x+02+1nzXVwfHz8vPlevWLt074+ZMfwd8RevTpgJyfsI79feTFf8iBNxixcpewFG7Lzc9bRmRfAl55J3yaM286Cbduvry+Y49vLKGFemnB/xtKQ2SkRW4ZJygZ9FofrRGfrhedztrHuk5ONlTi2zxNqYcsVdJtydvH+/Y+vL67fvmGrCKgQiThcBe5QC9LwrsU0vkljWwyyY85ibrs6C/gDj9k69tKUBy3joA3D3turwFmM2Tz2XKY53PM13MsJ04bsJQtSzz3pma2WzqgJqUPboN9qsQ0zTwcsXSDtRBB7W+w0XAcJCwMOfdswiAcs8e2pQfMY2/Mu8xIYy1kYe3MvsH3m00KMg2NlUUEYL6EJx5znazuB2fV8NfitNWHvTRbbSQrbS9Y2cHhzsmVJSMSK2QUvbPevtgNSQyn5XsDtmNGCYxdGz8JYyAsmimCuNffmi5SlIBCxtGtYc2T5piWnm/n2nIE0uZPifv0t7MpOmRMCPdhXytnSjiIvmE/gIV3gtnFvdjDnrkGCu1rAqtwx63XOTHbJbCf1HuzUC3HNnsPZMfG53EA6IYa3/3mfA2Y8eIk3BeUzgEHxls19a86XS2u5tK37oUamMBjoZ+x4cAYGgaaAHyOyY3vJjBWocGRt9cpbsITIAoY2vvcC/aBdf42yPThufq83NeQiOWCtA/ZZThXzOfQD/rIX0dfd/jcT2sLZmT5gx2cjac34AU7+xX7grAdaZy+hP4z0kFzCHDsIQnBgvmcLnQ1WyylHmiBGF977oUNyMdRJp7CqF7E131opeATxFEyzJ3Ack1Jv4NyL2MWmTahnj0nxuA0nYtfZQkdjUCQP9HWJ6ttWFC5Ze7/84vMJaLuXLpY89RzYzdL2wCKnXtr2XBCuBypkSD6q6wU2LkOX6/IZpSYf0YIm6gjJVmib5Dz84e3P16Crs9ieo2c8kd4LzQo5l4SzdA3q3o68iKPxueyu7YdhVGfdFGl0Al0+dYMSv2bYZZ041AH+YzNKdmTq3R47Hp3pZieXre8KVSFNeRF3dXZD+ng7aWw3qd0LdjT3qBn18Fayo9Kh4OGNopbYWXRfhg9KT8FhJ7XBL24mTT2Q8XmXreyS8DQyAi664TR6aeaO7PaK2vZMud3XsTQzLI7VbAX6c5LsAiLYHxNYVuyBxwanD7GHnC5D8YKvhWAnfL8BoXNqJ+ASc2pIAAMJF866TREedCMKYRyaINoZxU4YxSgILe070CsMaNucDCh4Gq8ccpFT7uMCgCTfgOckv6xEG5ze521aoBOCr4M+RrGeDvppG+Ktn3oRuMNwhtFZw2UAmZiDPnM2hKiYhESXb8BX4NrUKJyTKzx3O7JdF/tlc+aGmaxmM8/xMC6hsUDstucQIhiw3kcDN4STVOSTeZayHBs6KNZLK0oWfqN3wqfBhOUfwYhzJicAODDoCwrAF2MNbkTOpLitgqI5kTJJhwAVUBvC2Qy0VlAANhiJ8Hk9vfgrCAlT7naHGGi6vQ78y0wZJCDXPsABfZ31usqS84X7dsDlWnOGnMEIBDSkyYIRsWwa6qL9tEYLiYHHAkgRQbSfgqu/q0pitMd+R4ojlbMubdfwQyMRvoh4Rb1oDdkCYNa5H05BU3FyEILAIwREGJlHVZbdri7p9Sb1LQQoyJkXA36UkEZIRJgeEC08eAgAKAZ7AZ1Ox2xqx0ayDRwAj9wV4W+9CGERxAqDRd0uUE4XYC+00oWdHLhPoH5wUQYY9XR3m8T4Z72R0z3rG8Zw2B9yUJenML4cvRPay3YJYwADwF/EACxaTZlwHux76nvF0wxEnJA+EToF12W7APMAls3icMk+fff+9Z/fXFjffXz3xnzzaYIOpxh08ymnNR6j8zNdiwc24Cv30630NuI18DkMfT2L8FmIh9ZwFYEXyAEq+A4DMChwW+BclAdBd3sD7hC9UYF2C2oSkgbJaskJlEttUtDtk9BYgoU8nmVLpl3MLDDosWRd/i7x/JUFvqLWEIcRL16SsfdGOmjScbcPARyBJfOWkV+XBX68meSa+hI/PAL+pH7wlXZ4I4R/y8z2GxlUiE2AWWae77Pv3n74wKQsDluTgs4XsU3aKkxbAP1zBiDY8BIAZUuutdjRESiMOx7z4GE8frBjK0y0Q6kO703r48XV9duPh61ixKSgDDsoCH8u3pc3UX6Pn2Jb7812phu0FUlL7ohppaQnkyp6MEXkrcPyDOoKvxzs5aq7BZDsOe2EcpecqzYmlhR+m1j7453WKE9yiktbL7+JIeupvBJS18urzhlZ6SxVEix+BbnbnKfWDEwYF6cdQloDjRaAyMPWt7VxhdruGpz12EVB6Peu0diajyRX1B2RLzLBBsz9uo91A0Pw4SATVGHnV4tw5RfiIA2R2k9IB4wZ3O62wfy/zeoZJa8BgH1pnZ3KnRAK+1RV8U/oLAgDBQ9eHAYI+6WvQKc6CwoJZc5PO8JttFj7G/IhqgXQ/opU7liaZOFTL+3UWWRa/+k/bygfxA3dgoX+LL8Cdmev2F9uCOjCl/++/gRuj1b57ofrYUEN1DUBtOUAnEuY9sE0uuzaTu7YpS52ftwy2JWNThNsB7YPxpSo/v0KlguuHVPk+6HVAUu3P92y//nb32kuCLHtpf1XmOBP0Miuwgt2n2CY4ZC2IsD0sXayFTrQFzowONX7g/06ALuyXG8J+V/PVFTPC5reIkPkO2Imcf0jTwDZfq0BgP3OfxvHYfxNTQhK3i+lUXFI6Al0trbusWIly1Q6Va3wb/Z9q2fr1eUKdSaENLP9hCvG3CqkXQ2DircTgRiRS4n7YqEQValAk6DIfrr+mVEVoyBH6XqRFGPIWyXcnWSFG66m1A01HCRQUBNLaduA/WX1CIFPHin/cIOVgbXm+F4UbcfjNAytpR1sLTuer6hs2Lot2YnKcd9UuE3monAKOT9mR69RAMVblMSYvf6zyx88h0dpXGoT8tjVvtkzdvPE2O2uhrKeHjfq6fG/vp4CKtynps8Xc0XEtcX+fxTzbki5RwFy0bp8uppbdpLwOLX4/VeaXAuW4nUsxB8qDIdUW0Q4pvSiEn0JpdRoio1QQX8PTZEU9cz296I6K/KjBEmTez+lZPas26dkdo93pxqXnVgQw7VW5XV4Z4UQRcFvao+PGUfG47fBHMC6VloYrAGB4BqcHdbz1Th+aGA1Rmu1Wt8q+0a0qy0RFaBh0EPiiIdN9maTvdm2INpqT9oUbL5dox8KEp74FxClRtNT8eiu0eKfUJ/dlHTUuPEYUzYt17ZWq7J7pER1vAQI3ZRZfyQ5A/GfvcRnK39yrIfQc/Wm/omzp79IQod6F0vRZv/JtCcnG/6qRXi/qnewp/dxrbfMOp4xwa3CawidyskR8HooKyz4pNQ5EozyQ3GIlRdfshOkErmYRz6kOCIdliVILEgWBcgJVdWocvj1OdbyvBQRWFYb7BqFoqLbNURs14q3+Hk508sv6EwK1OyhUDs8nGL5a6F89K7bqg7GBId1G1o6le9HhWYqLa2ydQjGWYJx56xxZRWDoszAEkniecOiK90pgT7fm7xqCkm9tCTcppJeMnRitdHlASVapdGTcpDMpAXyoUXqJd6iz1Y42CryphzI/7SwYTWXMiFqxxxQIWUYY5YflhQl2zRsxp6yatnt6SN23O2edaVJg9eFRCNJk5qb55uIO6l2iLmuwKqHVac0hZVZonwFvD8CUGtdfTg7vTEMHGTRoNvKGGppGFOMMIxbxTUrwU6ZzlhSqpVoh1mJ+bBlOOEqSDVSZ0UIBVLHszNIMcN0kSH1AkWLMr5E2GB0eQVrCdGTByVy6hErFrt4KpA9cUpuLeYplc49PHmFtFeCdKMg9Fs2ZqobeyaBTZUAayCgSGX/Cjp1AuXpURNx59rh0gtkIbmnnCyi8R62fs0+dhyl7hT4/lU9dTbVtLZdtJ4+wPqHqZVPuX4d4/LyeOc36xCWFZGEYfvePOCusRwGw7vuoMT8QascQN/iwRd7Q+cc4Ng8spK80Id2NmxPt5C5SqJ0UkVmKUvhJWp0r0SYE5WHH8ArQbhMUjzmwtC7bIsaMdANQgK4diympT6J0ajxmeMp9pqkhjjWMB5Mg0pgxR6H7BjwM+xz1/FBvAoCyK6rJwf5a3lo0BlNh73O1DB6jmObpr3z0KAYWDsvKJrQp5sjE2szpjhchjdZJofQRDuonrWgr2fJwo540hJeKyvIJU6MnEBZ/fTx7b+9e//eury4fv3v8t5PEtZpwegPHy7+mNAVnvz4EELQUJb6CR5RMWltbxMsxk3xDFIViSw53xkQ5jBaieo1DsTqdHbIaYUzbdiqxuTKlYM0q65gnAUYJV0xKgZhKaW8olPy0UitsToty4gybGTXa7L7OCI0N1PDyzmAmXmEqM5e0BHNTJwNA1uADEQf70FEdJDBglM1E68HNVHDY+IobcNyohjyJXGcfHFyycStnQI0kqXAfrGmadRpEcfrtc9GDlPBm6erOIAxlTqQrF9QqoX5V0PlQqiPh9dLAtX1NFTzqzLt4fUW13MhPo9pR5L/AE3CJDthL1/iaiQGaD5czRf5GT+qfR6n0e/hnSyx5JMt+RQssWE9rYkaGAx4kQee7D61B6sATXe3dKUF1KQN/41G30HY6yRCzdyWvEelQfoPd2jag9OOYQwc2xmcnjb6j+rQkgepNqIP6Q+66EPw31lH9SFS0PJ4UHpKy+X3KxuA5y/cxUoAj3ngcNKdTKBU+FdqpJl2UimzfF54qcAxoY3yKmFxlwEvBXJBr3RFiOFFGwHLwVSdOEzwMs8qLpswllXVCwtgDX+4QS7cHhzXCpuZQYDrgamsfKrCMhDDXuGZmdT9uxam9vNoBV1E3pAZS5YJgK19pbo3xcSU46ur79/9NGaihpVdEKG6SBujmjgLaONZQFYoUW1JzDjJq390IJVtDHaFe3FAz7UjsegjyKEoCxmO4I/Zp3SKqoYqR9SBar2P3cFTXgbQlSJbAjqhF9U18VWW1fBLiYv43oKgAezDRwPSO4uuToKjf8mGk6KnKFex82LDcx/8nzMeS8AwHkudHI/xqGM8FigAvklF1Y7whCRPOIF+VoLpjiAF6xjdHFll1ZZ7zDv/g8MkGCqAC5HtgM3USFTHyVtlzxrLTgR6EiXYUJYRKE7Slg1nsQruEou8utbrl5RHXrTkG1AO18L6kUWXMrUjGnwDXsK8VbXkfk9nEyBJP+/9pdjSBnYieJfLS2UeprFGR+UBIjvcvgbT+TzQlNz4OFtz7f1GvALSffV1bcodjVrOTgBpVSIm3RUO7+Bfwf9SD8AKL8GTmsBfO2GrQb8i0ulqBhu6tB1wK+7lCs9RxuOAr6ULoB23jFWwju1IU3nhrkmLYLxBNzFznsh5ikGGG6VxaWTilEaqnNs7mooOizR0NZfi8tF9fW1qHwzbR4J68x5QA1aRH9qu3O+R5Ak8bco9K7utq8wzNr4pbxwPo+t0hKQbidwZmdHjrWCxYnej0+p0Iq+zXQsDmoVeNDFja8lCU319hXrt3twWE7zfMrgEvLRyhUlnQtBClOo2xWpzBy23mDno/CstQH6RB1zyKm2FAzX49/yFYJ79z1oGaS9mpQDrAnTujXqLSiqFBSnjVzcd4PWk4ghuK/1JOk/2pvndNFyQ+I6yiTJ+N1tbuT+yA5lS64uqMYu5CLi11qyAoARChLQentW1ytDa+MWLtCPfrL4GvdMeNfRarUdmG2mIMT4Bn3IOell8bSkl9UOJ4hC4ydRH/khAZC2yLI4D5a2dDE5QAZMOvSnjFFfxOJ0I4cXTFSxmyyCfBUXB6PenNQ9M47TdMU4vmYa6gfhEUKHLtkvuenjk3R+iC6GGRlQN8XIZIZqfcrwZp+LqWpNE1qNu17Q7I8M47Z3aI7c5M68PLmHrerM4R5G3+Qb6aCTRNd6r1aqniJfh5mt3G8irW/L4jM4VSweLBWb8r+AGbPGWJXdehPcPtAb4CGyVt2J++NHCvCThaSuv5Co3dDKsDhyeB2ECsFctMMuUWrmggMiGpetQUYvEyImh5ZwDsJR5KpfIXBbMKZOgWgEVIPJMFh6XEVgTB4CNleAcQO/AzxmMBKuFuQhiVs4F7qfQ9uhJ/PmIHPeGgNE1TfMw6Pe6EP/PWuwF4OFTCgIeJBVt1jXPxLdhqaKPJGEhq6EKRvHz+ACY+MYb3j4yUGd7hnkAyZFQljjeg+RiDUEnBvt6wFNUcO8ilr/E648pWw119iAifKs0K9qGBpzjfu0iQAvB4o12OAezsFbRoU62IZE3WI+A3RDxDt1wHRxKxE/2c1tN9skL2vGdjFD4qFUzduI48lSJww1d1niWDwD4a2/4DWJBwKI13AwTRNr9FKtsPv5EqXGuTZ2QdMoqlUfvESRNEj3DSl1rP1HXquKydRWYfatG38rYCjJTUD1w5iUzn0OkCpU2v2IBuxASLWDfwO1T2KNxrAIWLYrvZADaETzTKf3uzgQtRTjd9JzVcMKqrLrdS2CjzLbZOxtthjiTz2icqiE8mIqp6mzxqORwzk47DTxLvSVHB+Kb4v7Ho+KyZ4P+Uz67VuxCS7bQZEGPe7uKbXSIuavxCRzW9BFJ//4+QrRP9gH+PtVn8ww6m2fQ2T7Ro4Im93cuQ839fes4tOlT08X9x8c7xfe77P6VZNdUnq69qucfTfTQfSQpRHuOKcg7CPBUnwrCtdZUBlc9g3BJv3uH373D77L7P+odfrzTpHMwuG9HCR6+YU6QcCexAFlQLbTLBwCwhTeAZcPrSjb/pQGyYKZmrRDRIHjR6CZ7bQny50NFN7xIXOu1/9c+oKVt32xnl60oP/lyKzK/z/j3i6i/fJYrGhvdL5CGscf8l2OfaQnFe5f7qc0+j48N88uLwwYRQU5udLDeK9bezjbbAjbJx90/G8rKK3h8l1ppqGGCU/5dUfUnHez8H/jkREzXYIJddHnrll04Dvd5jL+6ZVcp9307Xi3zi154BoK5MSTDU1787hc/1+LEXR4W0281EvUIOv9tu/gF4JuPFx/EDwLxFIwmTw7+F3gXTM33RAAA"

NOTEBOOK_BUILD = "wave9-l2-raster-v1"
HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491400032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
MODEL_PATH = ""

TARGET_PREFILL_TPS = 15_000.0
FORCE_Q8_FOR_PTX = True
MICRO_REPEATS = 2
PRODUCTION_REPEATS = 2
COLD_ITERS = 3
WARMUP_ITERS = 3
MEASURE_ITERS = 10
RUN_NCU = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave9-{RUN_ID}"
META_REPO = ROOT / "meta"
BASE_DIR = ROOT / "wave4"
CAND_DIR = ROOT / "wave9"
RESULTS = ROOT / "results"
BASE_TARGET = ROOT / "target-wave4"
CAND_TARGET = ROOT / "target-wave9"
for path in (ROOT, RESULTS):
    path.mkdir(parents=True, exist_ok=False)

def run(cmd, cwd=None, env=None, timeout=7200, check=True):
    merged = dict(os.environ)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    proc = subprocess.run(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=merged,
        capture_output=True, text=True, errors="replace", timeout=timeout,
        stdin=subprocess.DEVNULL,
    )
    if check and proc.returncode:
        tail = (proc.stdout + "\n" + proc.stderr)[-5000:]
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(map(str, cmd))}\n{tail}")
    return proc

def save_log(name, proc):
    path = RESULTS / name
    path.write_text(
        f"$ {' '.join(map(str, proc.args))}\nexit={proc.returncode}\n\n"
        f"--- stdout ---\n{proc.stdout}\n--- stderr ---\n{proc.stderr}",
        encoding="utf-8",
    )
    return path

gpu_proc = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"], timeout=60)
gpu_rows = [line.strip() for line in gpu_proc.stdout.splitlines() if line.strip()]
if not gpu_rows:
    raise SystemExit("No NVIDIA GPU is visible. Enable a Kaggle GPU accelerator.")
print("Visible GPUs:")
for row in gpu_rows:
    print(" ", row)
first = [x.strip() for x in gpu_rows[0].split(",")]
if len(first) < 5 or "T4" not in first[1] or first[2] != "7.5":
    raise SystemExit(f"GPU 0 must be NVIDIA T4 compute capability 7.5; got: {gpu_rows[0]}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if shutil.which("cargo") is None:
    installer = run(["bash", "-lc", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], timeout=1200)
    save_log("rustup-install.log", installer)
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    raise SystemExit("Rust installation did not expose cargo.")

clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
save_log("git-clone.log", clone)
have_rev = run(["git", "cat-file", "-e", f"{BASE_REV}^{{commit}}"], cwd=META_REPO, check=False)
if have_rev.returncode:
    fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
    save_log("git-fetch-base.log", fetch)
run(["git", "worktree", "add", "--detach", BASE_DIR, BASE_REV], cwd=META_REPO)
run(["git", "worktree", "add", "--detach", CAND_DIR, BASE_REV], cwd=META_REPO)
if run(["git", "rev-parse", "HEAD"], cwd=BASE_DIR).stdout.strip() != BASE_REV:
    raise SystemExit("Baseline checkout mismatch")

def decode_patch(payload, expected_sha, name):
    raw = gzip.decompress(base64.b64decode(payload, validate=True))
    digest = hashlib.sha256(raw).hexdigest()
    if digest != expected_sha:
        raise SystemExit(f"Embedded {name} patch digest mismatch: {digest}")
    path = RESULTS / f"{name}.patch"
    path.write_bytes(raw)
    return path

wave3_patch = decode_patch(WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256, "wave3")
wave4_patch = decode_patch(WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256, "wave4")
wave9_patch = decode_patch(WAVE9_PATCH_GZIP_B64, WAVE9_PATCH_SHA256, "wave9")
for tree in (BASE_DIR, CAND_DIR):
    for patch in (wave3_patch, wave4_patch):
        run(["git", "apply", "--check", patch], cwd=tree)
        run(["git", "apply", "--whitespace=nowarn", patch], cwd=tree)
run(["git", "apply", "--check", wave9_patch], cwd=CAND_DIR)
run(["git", "apply", "--whitespace=nowarn", wave9_patch], cwd=CAND_DIR)
run(["git", "apply", "--check", "--reverse", wave9_patch], cwd=CAND_DIR)

base_main = (BASE_DIR / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
cand_main = (CAND_DIR / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
base_sm75 = (BASE_DIR / "glcuda/src/kernels/glcuda_sm75.ptx").read_text(encoding="ascii")
cand_sm75 = (CAND_DIR / "glcuda/src/kernels/glcuda_sm75.ptx").read_text(encoding="ascii")
cand_mod = (CAND_DIR / "glcuda/src/kernels/mod.rs").read_text(encoding="utf-8")
cand_runner = (CAND_DIR / "glcuda/src/runner.rs").read_text(encoding="utf-8")
cand_bench = (CAND_DIR / "glcuda/examples/bench.rs").read_text(encoding="utf-8")
cand_parity = (CAND_DIR / "glcuda/tests/parity.rs").read_text(encoding="utf-8")
r256_entry = ".visible .entry gl_gemm_mma_q8_r256("
base_grid = base_sm75[:base_sm75.index(r256_entry)]
cand_grid = cand_sm75[:cand_sm75.index(r256_entry)]
markers = {
    "main_ptx_unchanged": base_main == cand_main,
    "r256_unchanged": base_sm75[base_sm75.index(r256_entry):] == cand_sm75[cand_sm75.index(r256_entry):],
    "wave4_raster_params": base_grid.count(".param .u32 p_l2_raster"),
    "wave9_raster_params": cand_grid.count(".param .u32 p_l2_raster"),
    "wave9_ctaid_x": cand_grid.count("%ctaid.x"),
    "wave9_ctaid_y": cand_grid.count("%ctaid.y"),
    "wave9_predicated_swaps": cand_grid.count("@%p_l2 mov.u32"),
    "mma_count_unchanged": base_grid.count("mma.sync.aligned.m8n8k16") == cand_grid.count("mma.sync.aligned.m8n8k16") == 16,
    "barriers_unchanged": base_grid.count("bar.sync 0") == cand_grid.count("bar.sync 0") == 2,
    "shared_unchanged": sum(line.lstrip().startswith(".shared") for line in base_grid.splitlines()) == sum(line.lstrip().startswith(".shared") for line in cand_grid.splitlines()) == 2,
    "host_env_reads": cand_mod.count('var_os("GLCUDA_L2_RASTER")'),
    "host_launch_methods": cand_mod.count("pub fn gemm_mma_q8_l2("),
    "production_dispatches": cand_runner.count("l2_raster_enabled()"),
    "bench_markers": cand_bench.count("[gemm-l2-raster"),
    "bit_parity_tests": cand_parity.count("fn gemm_mma_q8_l2_raster_is_bit_identical()"),
    "rejected_wave5678_markers": sum(x.lower().count(t) for x in (cand_main, cand_sm75, cand_runner, cand_mod) for t in ("w8pc", "glcuda_r128", "gl_gemm_mma_q8_r128", "q8_rowcta")),
    "cp_async_instructions": sum(line.lstrip().startswith("cp.async") for text in (cand_main, cand_sm75) for line in text.splitlines()),
}
expected = {
    "main_ptx_unchanged": True, "r256_unchanged": True,
    "wave4_raster_params": 0, "wave9_raster_params": 1,
    "wave9_ctaid_x": 2, "wave9_ctaid_y": 2, "wave9_predicated_swaps": 2,
    "mma_count_unchanged": True, "barriers_unchanged": True, "shared_unchanged": True,
    "host_env_reads": 1, "host_launch_methods": 1, "production_dispatches": 1,
    "bench_markers": 1, "bit_parity_tests": 1,
    "rejected_wave5678_markers": 0, "cp_async_instructions": 0,
}
if markers != expected:
    raise SystemExit(f"Wave 9 structural markers changed:\nexpected={expected}\nactual={markers}")
for name, text in (("main", cand_main), ("sm75", cand_sm75)):
    if "\r" in text or not text.isascii():
        raise SystemExit(f"Candidate {name} PTX must remain ASCII with LF endings.")

print(f"Notebook  {NOTEBOOK_BUILD}")
print(f"Baseline  Wave 3 {WAVE3_PATCH_SHA256} + Wave 4 {WAVE4_PATCH_SHA256}")
print(f"Candidate Wave 9 {WAVE9_PATCH_SHA256}")
print(f"Run root  {ROOT}")
print("Structural markers:", markers)


## 2 - Fetch the pinned production model


In [ ]:
print(f"MODEL FETCH START [{NOTEBOOK_BUILD}]")
sys.stdout.flush()

import urllib.error
import urllib.request

MODEL_CACHE = WORK / "models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
model_dest = MODEL_CACHE / HF_FILENAME
model_part = model_dest.with_name(model_dest.name + ".part")
model_url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def park_invalid(path, reason):
    parked = path.with_name(path.name + f".invalid-{reason}-{RUN_ID}")
    path.replace(parked)
    print(f"Parked invalid cache file: {parked}")

def validate_model(path):
    if not path.is_file():
        return False, "missing"
    size = path.stat().st_size
    if size != HF_EXPECTED_BYTES:
        return False, f"size-{size}"
    with path.open("rb") as handle:
        if handle.read(4) != b"GGUF":
            return False, "magic"
    digest = file_sha256(path)
    if digest != HF_EXPECTED_SHA256:
        return False, f"sha256-{digest[:12]}"
    return True, digest

valid, detail = validate_model(model_dest)
if valid:
    print(f"Reusing verified model: {model_dest}")
else:
    if model_dest.exists():
        park_invalid(model_dest, detail)
    if model_part.exists() and model_part.stat().st_size > HF_EXPECTED_BYTES:
        park_invalid(model_part, f"oversize-{model_part.stat().st_size}")
    if model_part.exists() and model_part.stat().st_size == HF_EXPECTED_BYTES:
        partial_valid, partial_detail = validate_model(model_part)
        if partial_valid:
            model_part.replace(model_dest)
        else:
            park_invalid(model_part, partial_detail)
    if not model_dest.exists():
        start = model_part.stat().st_size if model_part.exists() else 0
        headers = {"User-Agent": "GwenLand-Wave6-Kaggle/1.0"}
        if start:
            headers["Range"] = f"bytes={start}-"
            print(f"Resuming model download at {start / 2**20:.1f} MiB")
        else:
            print(f"Downloading {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}")
        request = urllib.request.Request(model_url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
        except urllib.error.HTTPError as exc:
            raise SystemExit(f"Model download HTTP {exc.code}: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        except urllib.error.URLError as exc:
            raise SystemExit(f"Model download connection failed: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        status = getattr(response, "status", response.getcode())
        if start and status != 206:
            print(f"Server ignored Range (HTTP {status}); restarting the partial download.")
            start = 0
        mode = "ab" if start and status == 206 else "wb"
        downloaded = start
        last_print = time.monotonic()
        with response, model_part.open(mode) as output:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                output.write(block)
                downloaded += len(block)
                now = time.monotonic()
                if now - last_print >= 5:
                    pct = 100.0 * downloaded / HF_EXPECTED_BYTES
                    print(f"  {downloaded / 2**20:.1f} / {HF_EXPECTED_BYTES / 2**20:.1f} MiB ({pct:.1f}%)")
                    last_print = now
        part_valid, part_detail = validate_model(model_part)
        if not part_valid:
            park_invalid(model_part, part_detail)
            raise SystemExit(f"Downloaded GGUF failed integrity validation: {part_detail}")
        model_part.replace(model_dest)
valid, digest = validate_model(model_dest)
if not valid:
    raise SystemExit(f"Final GGUF validation failed: {digest}")
MODEL_PATH = str(model_dest)
MODEL_FETCH = {
    "repo": HF_REPO,
    "revision": HF_REVISION,
    "filename": HF_FILENAME,
    "url": model_url,
    "bytes": model_dest.stat().st_size,
    "sha256": digest,
    "path": MODEL_PATH,
}
(RESULTS / "model-fetch.json").write_text(json.dumps(MODEL_FETCH, indent=2), encoding="utf-8")
print(json.dumps(MODEL_FETCH, indent=2))


## 3 - PTX assembly, spill, and occupancy gate


In [ ]:
ptxas = shutil.which("ptxas")
if ptxas is None:
    candidates = sorted(Path("/usr/local").glob("cuda*/bin/ptxas"), reverse=True)
    ptxas = str(candidates[0]) if candidates else None
if ptxas is None:
    raise SystemExit("ptxas is required for Wave 9 but was not found in the Kaggle image.")

tool_versions = {}
for name, cmd in {"nvidia_smi": ["nvidia-smi"], "rustc": ["rustc", "--version", "--verbose"], "cargo": ["cargo", "--version"], "ptxas": [ptxas, "--version"]}.items():
    p = run(cmd, check=False, timeout=120)
    tool_versions[name] = (p.stdout + p.stderr).strip()
    print(f"--- {name} ---\n{tool_versions[name][:1500]}")

def parse_ptxas_function_resources(text, fn):
    header = re.search(rf"(?m)^ptxas info\s*: Function properties for {re.escape(fn)}\s*$", text)
    if not header:
        raise ValueError(f"missing Function properties header for {fn}")
    tail = text[header.end():]
    next_function = re.search(r"(?m)^ptxas info\s*: Compiling entry function\b", tail)
    block = tail[:next_function.start()] if next_function else tail
    registers = re.search(r"Used\s+(\d+)\s+registers", block)
    spills = [int(x) for x in re.findall(r"(\d+) bytes spill (?:stores|loads)", block)]
    smem = re.search(r"(\d+)\s+bytes smem", block)
    if not registers or len(spills) != 2:
        raise ValueError(f"missing register/spill counters for {fn}")
    return {"registers": int(registers.group(1)), "smem_bytes": int(smem.group(1)) if smem else 0, "spill_bytes": spills}

def assemble_module(label, src, module_name, functions):
    cubin = RESULTS / f"{label}-{Path(module_name).stem}.cubin"
    ptx_file = src / "glcuda/src/kernels" / module_name
    p = run([ptxas, "-arch=sm_75", "-v", ptx_file, "-o", cubin], cwd=src, timeout=600, check=False)
    save_log(f"ptxas-{label}-{Path(module_name).stem}.log", p)
    text = p.stdout + "\n" + p.stderr
    print(f"\n--- ptxas {label}/{module_name} ---\n{text}")
    if p.returncode:
        raise SystemExit(f"ptxas failed for {label}/{module_name}")
    resources = {fn: parse_ptxas_function_resources(text, fn) for fn in functions}
    for fn, res in resources.items():
        if any(res["spill_bytes"]):
            raise SystemExit(f"PTXAS spill gate failed for {label}/{fn}: {res}")
    return {"cubin": str(cubin), "resources": resources, "log": text}

PTXAS = {}
for label, src in (("wave4", BASE_DIR), ("wave9", CAND_DIR)):
    PTXAS[label] = {
        "main": assemble_module(label, src, "glcuda.ptx", ("gl_quantize_q8", "gl_attn_decode_rows_f32", "gl_attn_rows_probe")),
        "sm75": assemble_module(label, src, "glcuda_sm75.ptx", ("gl_gemm_mma_q8", "gl_gemm_mma_q8_r256")),
    }
for fn in ("gl_quantize_q8", "gl_attn_decode_rows_f32", "gl_attn_rows_probe"):
    if PTXAS["wave4"]["main"]["resources"][fn] != PTXAS["wave9"]["main"]["resources"][fn]:
        raise SystemExit(f"Unchanged main-PTX resource changed for {fn}")
if PTXAS["wave4"]["sm75"]["resources"]["gl_gemm_mma_q8_r256"] != PTXAS["wave9"]["sm75"]["resources"]["gl_gemm_mma_q8_r256"]:
    raise SystemExit("Wave 9 changed r256 resources")
base_res = PTXAS["wave4"]["sm75"]["resources"]["gl_gemm_mma_q8"]
cand_res = PTXAS["wave9"]["sm75"]["resources"]["gl_gemm_mma_q8"]
if cand_res["smem_bytes"] != base_res["smem_bytes"] or any(cand_res["spill_bytes"]):
    raise SystemExit(f"Wave 9 shared/spill gate failed: base={base_res}, candidate={cand_res}")
resident_ctas = min(4, 65536 // (256 * max(cand_res["registers"], 1)), 65536 // max(cand_res["smem_bytes"], 1))
resident_warps = resident_ctas * 8
if resident_warps < 24:
    raise SystemExit(f"Wave 9 occupancy gate failed: {cand_res}, {resident_warps} warps")
RASTER_RESOURCES = {"wave4": base_res, "wave9": cand_res, "resident_ctas": resident_ctas, "resident_warps": resident_warps}
PTXAS_OK = True
print("\nPTXAS resource gate PASS")
print(json.dumps(RASTER_RESOURCES, indent=2))


## 4 - Hardware, bit-parity, and model correctness gates


In [ ]:
def cargo_env(target):
    return {"CARGO_TARGET_DIR": str(target), "RUST_BACKTRACE": "1"}

def cargo_run(label, src, target, args, log_name, timeout=7200):
    p = run(["cargo", *args], cwd=src, env=cargo_env(target), timeout=timeout, check=False)
    save_log(log_name, p)
    hay = p.stdout + "\n" + p.stderr
    if p.returncode:
        raise SystemExit(f"{label} failed; see {log_name}")
    return hay

for label, src, target in (("wave4", BASE_DIR, BASE_TARGET), ("wave9", CAND_DIR, CAND_TARGET)):
    cargo_run(f"{label}/check", src, target, ["check", "--locked", "-p", "glcuda"], f"check-{label}.log")
    cargo_run(f"{label}/lib", src, target, ["test", "--locked", "-p", "glcuda", "--lib"], f"test-{label}-lib.log")
    hay = cargo_run(
        f"{label}/parity", src, target,
        ["test", "--locked", "-p", "glcuda", "--test", "parity", "--", "--test-threads=1", "--nocapture"],
        f"test-{label}-parity.log",
    )
    if "SKIP: no CUDA driver/device" in hay:
        raise SystemExit(f"{label} parity suite skipped CUDA instead of running on T4")

exact = cargo_run(
    "wave9/l2-raster-bit-parity", CAND_DIR, CAND_TARGET,
    ["test", "--locked", "-p", "glcuda", "--test", "parity", "gemm_mma_q8_l2_raster_is_bit_identical", "--", "--exact", "--nocapture", "--test-threads=1"],
    "test-wave9-l2-raster-bit-parity.log",
)
if "gemm_mma_q8_l2_raster_is_bit_identical ... ok" not in exact:
    raise SystemExit("Wave 9 exact raster parity test did not execute successfully")

for label, src, target in (("wave4", BASE_DIR, BASE_TARGET), ("wave9", CAND_DIR, CAND_TARGET)):
    cargo_run(f"{label}/bench build", src, target, ["build", "--locked", "--release", "-p", "glcuda", "--example", "bench"], f"build-{label}-bench.log")
    cargo_run(f"{label}/glbench build", src, target, ["build", "--locked", "--release", "-p", "glbench"], f"build-{label}-glbench.log")

BINS = {
    "wave4": {"bench": BASE_TARGET / "release/examples/bench", "glbench": BASE_TARGET / "release/glbench"},
    "wave9": {"bench": CAND_TARGET / "release/examples/bench", "glbench": CAND_TARGET / "release/glbench"},
}
for arm, bins in BINS.items():
    for kind, path in bins.items():
        if not path.exists():
            raise SystemExit(f"Missing {arm}/{kind} binary: {path}")

CORRECTNESS_OK = True
print("\nCorrectness gate passed for Wave 4 and Wave 9 on the real T4.")


## 5 - Diagnostic L2-raster GEMM microbenchmark


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")
samples = {"gate_up": [], "down": []}
pattern = re.compile(r"\[gemm-l2-raster (gate_up|down)\] ntok=244 grid ([0-9.]+) us \| grouped ([0-9.]+) us \| delta ([+-][0-9.]+)%")
for repeat in range(MICRO_REPEATS):
    p = run([BINS["wave9"]["bench"]], cwd=CAND_DIR, env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=7200, check=False)
    save_log(f"bench-wave9-{repeat}.log", p)
    if p.returncode:
        raise SystemExit(f"Wave 9 diagnostic bench repeat {repeat} failed")
    found = pattern.findall(p.stdout + "\n" + p.stderr)
    if len(found) != 2:
        raise SystemExit(f"Wave 9 bench emitted {len(found)} L2-raster rows, expected 2")
    for label, grid_us, grouped_us, delta_pct in found:
        samples[label].append({"grid_us": float(grid_us), "grouped_us": float(grouped_us), "delta_pct": float(delta_pct)})
DIAGNOSTIC = {}
for label, rows in samples.items():
    DIAGNOSTIC[label] = {
        "samples": rows,
        "grid_median_us": statistics.median(x["grid_us"] for x in rows),
        "grouped_median_us": statistics.median(x["grouped_us"] for x in rows),
    }
    DIAGNOSTIC[label]["median_delta"] = DIAGNOSTIC[label]["grouped_median_us"] / DIAGNOSTIC[label]["grid_median_us"] - 1.0
print(json.dumps(DIAGNOSTIC, indent=2))


## 6 - Production glbench A/B


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

model = Path(MODEL_PATH)
if not model.is_file() or model.stat().st_size < 10_000_000:
    raise SystemExit(f"Model path is not a plausible GGUF: {model}")

prompt_unit = (
    "Measure this deterministic systems prompt carefully. Explain how token-parallel "
    "integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
)
FIXED_PROMPT = prompt_unit * 8
arms = [
    ("wave4_attn_dsmem", "wave4", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1"}),
    ("wave9_l2_raster", "wave9", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_L2_RASTER": "1"}),
]
arm_map = {label: (build, env) for label, build, env in arms}

def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    if lo == hi:
        return values[lo]
    return values[lo] * (hi - index) + values[hi] * (index - lo)

def session_stats(path, expected_iters=MEASURE_ITERS, require_exact_oracle=False):
    data = json.loads(path.read_text(encoding="utf-8"))
    engine_blob = json.dumps(data.get("engine", {}), sort_keys=True).lower()
    if "glcuda" not in engine_blob:
        raise RuntimeError(f"Session did not record glcuda engine: {data.get('engine')}")
    validation = data.get("validation") or {}
    findings = validation.get("findings", [])
    parity = [f for f in findings if f.get("check") == "parity"]
    other_errors = [f for f in findings if f.get("severity") == "error" and f.get("check") != "parity"]
    if other_errors:
        raise RuntimeError(f"Session failed non-parity validation: {other_errors}")
    if not parity:
        raise RuntimeError("Session did not record the requested glproc oracle check.")
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", ""))
    if not match:
        raise RuntimeError(f"Could not parse glproc oracle evidence: {parity[-1]}")
    oracle_prefix, oracle_compared = map(int, match.groups())
    oracle_exact = oracle_compared > 0 and oracle_prefix == oracle_compared
    if require_exact_oracle and not oracle_exact:
        raise RuntimeError(f"Retained baseline lost exact glproc parity: {parity[-1]}")
    iterations = data.get("measurements", {}).get("iterations", [])
    prefill_ms = [float(x.get("prefill_ms", 0.0)) for x in iterations]
    decode_ms = [float(x.get("decode_ms", 0.0)) for x in iterations]
    prompt_counts = [int(x.get("prompt_tokens", 0)) for x in iterations]
    if len(iterations) != expected_iters or any(x <= 0 for x in prefill_ms + decode_ms):
        raise RuntimeError(f"Expected {expected_iters} valid iterations, got {iterations}")
    if len(set(prompt_counts)) != 1 or prompt_counts[0] <= 0:
        raise RuntimeError(f"Prompt token count changed: {prompt_counts}")
    prefill_tps = [prompt_counts[0] * 1000.0 / x for x in prefill_ms]
    decode_tps = [1000.0 / x for x in decode_ms]
    return {
        "prompt_tokens": prompt_counts[0],
        "prefill_tps_samples": prefill_tps,
        "prefill_mean": statistics.mean(prefill_tps),
        "prefill_p50": percentile(prefill_tps, 0.50),
        "prefill_p10": percentile(prefill_tps, 0.10),
        "prefill_latency_p50_ms": percentile(prefill_ms, 0.50),
        "prefill_latency_p95_ms": percentile(prefill_ms, 0.95),
        "prefill_latency_p99_ms": percentile(prefill_ms, 0.99),
        "decode_p50": percentile(decode_tps, 0.50),
        "validation_passed": bool(validation.get("passed", False)),
        "oracle_matching_prefix": oracle_prefix,
        "oracle_compared": oracle_compared,
        "oracle_exact": oracle_exact,
    }

PROD_RECORDS = []
for repeat in range(PRODUCTION_REPEATS):
    rotated = arms[repeat % len(arms):] + arms[:repeat % len(arms)]
    for label, build_arm, extra_env in rotated:
        archive = RESULTS / f"glbench-{repeat}-{label}.json"
        cmd = [
            BINS[build_arm]["glbench"], "run", "--engine", "glcuda",
            "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
            "--cold-iters", str(COLD_ITERS), "--warmup", str(WARMUP_ITERS),
            "--iters", str(MEASURE_ITERS), "--temperature", "0", "--seed", "42",
            "--kind", "prefill", "--verify-against", "glproc", "--out", archive,
        ]
        src = BASE_DIR if build_arm == "wave4" else CAND_DIR
        p = run(cmd, cwd=src, env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
        save_log(f"glbench-{repeat}-{label}.log", p)
        hay = p.stdout + "\n" + p.stderr
        if p.returncode:
            raise SystemExit(f"glbench {label} repeat {repeat} failed.")
        expected_banners = ["2-D token-grid prefill GEMM enabled", "dynamic-shared prefill attention enabled"]
        if build_arm == "wave4":
            expected_banners += ["GLCUDA_FORCE_Q8:"]
            forbidden = ["Q8_0 row-CTA prefill quantizer enabled", "GLCUDA_W8PC:", "W8PC MMA enabled", "r128 prefill GEMM enabled"]
        else:
            expected_banners += ["GLCUDA_FORCE_Q8:", "L2-grouped GEMM raster enabled"]
            forbidden = ["Q8_0 row-CTA prefill quantizer enabled", "GLCUDA_W8PC:", "W8PC MMA enabled", "r128 prefill GEMM enabled"]
        for banner in expected_banners:
            if banner not in hay:
                raise SystemExit(f"{label} did not confirm required dispatch banner: {banner}")
        for banner in forbidden:
            if banner in hay:
                raise SystemExit(f"{label} unexpectedly enabled: {banner}")
        if "r256 prefill GEMM enabled" in hay:
            raise SystemExit(f"{label} accidentally enabled r256.")
        stats = session_stats(archive, require_exact_oracle=True)
        rec = {"repeat": repeat, "arm": label, "archive": str(archive), **stats}
        PROD_RECORDS.append(rec)
        print(
            f"{label:18s} repeat {repeat}: P50 {stats['prefill_p50']:.1f} tok/s | "
            f"mean {stats['prefill_mean']:.1f} | P95 latency {stats['prefill_latency_p95_ms']:.2f} ms | "
            f"decode P50 {stats['decode_p50']:.1f} | oracle "
            f"{stats['oracle_matching_prefix']}/{stats['oracle_compared']}"
        )

PROD_SUMMARY = []
for label, _, _ in arms:
    rows = [x for x in PROD_RECORDS if x["arm"] == label]
    PROD_SUMMARY.append({
        "arm": label,
        "session_p50_median": statistics.median(x["prefill_p50"] for x in rows),
        "session_mean_median": statistics.median(x["prefill_mean"] for x in rows),
        "session_p95_latency_median_ms": statistics.median(x["prefill_latency_p95_ms"] for x in rows),
        "decode_p50_median": statistics.median(x["decode_p50"] for x in rows),
        "sessions": len(rows),
        "oracle_prefixes": [f"{x['oracle_matching_prefix']}/{x['oracle_compared']}" for x in rows],
    })

base_rows = [x for x in PROD_RECORDS if x["arm"] == "wave4_attn_dsmem"]
cand_rows = [x for x in PROD_RECORDS if x["arm"] == "wave9_l2_raster"]
paired = []
for repeat in range(PRODUCTION_REPEATS):
    a = next(x for x in base_rows if x["repeat"] == repeat)
    b = next(x for x in cand_rows if x["repeat"] == repeat)
    paired.append({
        "repeat": repeat,
        "prefill_p50_delta": b["prefill_p50"] / a["prefill_p50"] - 1.0,
        "prefill_mean_delta": b["prefill_mean"] / a["prefill_mean"] - 1.0,
        "p95_latency_delta": b["prefill_latency_p95_ms"] / a["prefill_latency_p95_ms"] - 1.0,
        "decode_p50_delta": b["decode_p50"] / a["decode_p50"] - 1.0,
    })
base_tps = statistics.median(x["prefill_p50"] for x in base_rows)
cand_tps = statistics.median(x["prefill_p50"] for x in cand_rows)
WAVE9_DECISION = {
    "baseline_arm": "wave4_attn_dsmem",
    "candidate_arm": "wave9_l2_raster",
    "baseline_tps": base_tps,
    "candidate_tps": cand_tps,
    "median_delta": cand_tps / base_tps - 1.0,
    "paired": paired,
    "correctness_green": bool(CORRECTNESS_OK),
}
WAVE9_DECISION["prefill_p50_pass"] = all(x["prefill_p50_delta"] >= 0.05 for x in paired)
WAVE9_DECISION["prefill_mean_pass"] = all(x["prefill_mean_delta"] >= 0.05 for x in paired)
WAVE9_DECISION["tail_pass"] = all(x["p95_latency_delta"] <= 0.05 for x in paired)
WAVE9_DECISION["decode_pass"] = all(x["decode_p50_delta"] >= -0.05 for x in paired)
WAVE9_DECISION["oracle_parity_pass"] = all(x["oracle_exact"] for x in cand_rows)
WAVE9_DECISION["retain"] = all([
    WAVE9_DECISION["median_delta"] >= 0.05,
    WAVE9_DECISION["correctness_green"],
    WAVE9_DECISION["oracle_parity_pass"],
    WAVE9_DECISION["prefill_p50_pass"],
    WAVE9_DECISION["prefill_mean_pass"],
    WAVE9_DECISION["tail_pass"],
    WAVE9_DECISION["decode_pass"],
])
print("\nProduction summary:", json.dumps(PROD_SUMMARY, indent=2))
print("\nWave 9 decision:", json.dumps(WAVE9_DECISION, indent=2))

TELEMETRY = {}
for label, build_arm, extra_env in arms:
    archive = RESULTS / f"telemetry-{label}.json"
    cmd = [
        BINS[build_arm]["glbench"], "run", "--engine", "glcuda",
        "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
        "--cold-iters", "0", "--warmup", "3", "--iters", "1",
        "--temperature", "0", "--seed", "42", "--kind", "prefill", "--verify-against", "glproc", "--out", archive,
    ]
    src = BASE_DIR if build_arm == "wave4" else CAND_DIR
    p = run(cmd, cwd=src, env={"CUDA_VISIBLE_DEVICES": "0", **extra_env, "GLCUDA_TELEMETRY": "1"}, timeout=14400, check=False)
    save_log(f"telemetry-{label}.log", p)
    if p.returncode:
        raise SystemExit(f"Telemetry run failed for {label}.")
    data = json.loads(archive.read_text(encoding="utf-8"))
    stages = (((data.get("telemetry") or {}).get("prefill") or {}).get("stages") or [])
    if not stages:
        raise SystemExit(f"Telemetry produced no stages for {label}.")
    TELEMETRY[label] = {"stages": stages, "session": session_stats(archive, expected_iters=1, require_exact_oracle=True)}

cand_stages = TELEMETRY["wave9_l2_raster"]["stages"]
stage_total_ms = sum(float(x.get("total_ms") or 0.0) for x in cand_stages)
attention_ms = sum(float(x.get("total_ms") or 0.0) for x in cand_stages if x.get("name") == "attention")
gemm_names = {"qkv", "attn_out", "ffn_gate_up", "ffn_down"}
gemm_ms = sum(float(x.get("total_ms") or 0.0) for x in cand_stages if x.get("name") in gemm_names)
prompt_tokens = cand_rows[0]["prompt_tokens"]
TARGET_ANALYSIS = {
    "prompt_tokens": prompt_tokens,
    "measured_tps": cand_tps,
    "target_tps": TARGET_PREFILL_TPS,
    "measured_prefill_ms": prompt_tokens * 1000.0 / cand_tps,
    "target_prefill_ms": prompt_tokens * 1000.0 / TARGET_PREFILL_TPS,
    "required_speedup": TARGET_PREFILL_TPS / cand_tps,
    "attention_share": attention_ms / stage_total_ms,
    "gemm_share": gemm_ms / stage_total_ms,
    "infinite_attention_ceiling_tps": cand_tps / (1.0 - attention_ms / stage_total_ms),
    "infinite_gemm_ceiling_tps": cand_tps / (1.0 - gemm_ms / stage_total_ms),
}

for repeat in range(PRODUCTION_REPEATS):
    b = RESULTS / f"glbench-{repeat}-wave4_attn_dsmem.json"
    c = RESULTS / f"glbench-{repeat}-wave9_l2_raster.json"
    p = run([BINS["wave9"]["glbench"], "compare", b, c], cwd=CAND_DIR, timeout=600, check=False)
    save_log(f"compare-{repeat}-wave4-vs-wave9.log", p)
    if p.returncode:
        raise SystemExit("glbench compare failed for Wave 4 vs Wave 9.")
PROD_OK = True


## 7 - Optional Nsight Compute evidence


In [ ]:
NCU = {"available": False, "runs": {}}
ncu = shutil.which("ncu")
if not RUN_NCU:
    print("Nsight Compute disabled by configuration.")
elif ncu is None:
    print("Nsight Compute CLI is not installed in this Kaggle image.")
else:
    listed = run([ncu, "--list-sections"], timeout=300, check=False)
    save_log("ncu-list-sections.log", listed)
    available_text = listed.stdout + "\n" + listed.stderr
    wanted = ["LaunchStats", "Occupancy", "SpeedOfLight", "WarpStateStats", "MemoryWorkloadAnalysis", "ComputeWorkloadAnalysis"]
    sections = [name for name in wanted if name in available_text]
    NCU["available"] = True
    NCU["sections"] = sections
    for label, build_arm, extra_env in arms:
        report = RESULTS / f"ncu-{label}"
        kernel_name = "regex:.*gl_gemm_mma_q8$"
        cmd = [ncu, "--target-processes", "all",
               "--kernel-name", kernel_name, "--launch-count", "1",
               "--force-overwrite", "--export", report]
        for section in sections:
            cmd += ["--section", section]
        if not sections:
            cmd += ["--set", "basic"]
        archive = RESULTS / f"ncu-session-{label}.json"
        cmd += [BINS[build_arm]["glbench"], "run", "--engine", "glcuda",
                "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
                "--cold-iters", "0", "--warmup", "0", "--iters", "1",
                "--temperature", "0", "--seed", "42", "--kind", "prefill", "--out", archive]
        src = BASE_DIR if build_arm == "wave4" else CAND_DIR
        p = run(cmd, cwd=src, env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
        save_log(f"ncu-{label}.log", p)
        text = p.stdout + "\n" + p.stderr
        permitted = p.returncode == 0 and "ERR_NVGPUCTRPERM" not in text
        NCU["runs"][label] = {"returncode": p.returncode, "permitted": permitted}
        print(f"{label:18s}: exit={p.returncode}, counters={'captured' if permitted else 'unavailable'}")
        if permitted:
            imported = run([ncu, "--import", str(report) + ".ncu-rep", "--page", "details", "--csv"], timeout=1800, check=False)
            save_log(f"ncu-{label}-details.csv", imported)


## 8 - Package the Wave 9 evidence


In [ ]:
ptxas_resources = {arm: {module: data["resources"] for module, data in modules.items()} for arm, modules in PTXAS.items()}
manifest = {
    "schema": "gwenland.glcuda.t4-ceiling.wave9.fetch.v1", "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook_build": NOTEBOOK_BUILD, "gpu_rows": gpu_rows, "repo_url": REPO_URL, "base_rev": BASE_REV,
    "wave3_patch_sha256": WAVE3_PATCH_SHA256, "wave4_patch_sha256": WAVE4_PATCH_SHA256, "wave9_patch_sha256": WAVE9_PATCH_SHA256,
    "markers": markers, "tool_versions": tool_versions, "ptxas_ok": bool(globals().get("PTXAS_OK")),
    "ptxas_resources": ptxas_resources, "raster_resources": globals().get("RASTER_RESOURCES"),
    "correctness_ok": bool(globals().get("CORRECTNESS_OK")), "production_ok": bool(globals().get("PROD_OK")),
    "target_prefill_tps": TARGET_PREFILL_TPS, "production_repeats": PRODUCTION_REPEATS,
    "cold_iters": COLD_ITERS, "warmup_iters": WARMUP_ITERS, "measure_iters": MEASURE_ITERS,
    "model_fetch": globals().get("MODEL_FETCH"), "diagnostic": globals().get("DIAGNOSTIC"),
    "production_summary": globals().get("PROD_SUMMARY", []), "wave9_decision": globals().get("WAVE9_DECISION"),
    "telemetry": globals().get("TELEMETRY", {}), "target_analysis": globals().get("TARGET_ANALYSIS"), "ncu": globals().get("NCU", {}),
}
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
report = ["# glcuda T4 Ceiling - Wave 9", "", f"- notebook: {NOTEBOOK_BUILD}", f"- GPU: {gpu_rows[0]}",
    f"- baseline revision: {BASE_REV}", f"- Wave 3 patch: {WAVE3_PATCH_SHA256}", f"- retained Wave 4 patch: {WAVE4_PATCH_SHA256}",
    "- rejected Wave 5/Wave 6/Wave 7/Wave 8 patches applied: NO", f"- Wave 9 patch: {WAVE9_PATCH_SHA256}",
    f"- model: {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}", f"- model SHA-256: {manifest['model_fetch']['sha256']}",
    f"- ptxas: {'PASS' if manifest['ptxas_ok'] else 'FAIL'}", f"- hardware bit/reference correctness: {'PASS' if manifest['correctness_ok'] else 'FAIL'}",
    f"- production glbench: {'COMPLETE' if manifest['production_ok'] else 'PENDING'}", f"- target: {TARGET_PREFILL_TPS:.0f} prefill tok/s", "",
    "## Raster resource/contract gate", "", f"- Wave 4 grid64: {manifest['raster_resources']['wave4']}",
    f"- Wave 9 L2 raster: {manifest['raster_resources']['wave9']}",
    f"- projected resident CTAs/warps: {manifest['raster_resources']['resident_ctas']} / {manifest['raster_resources']['resident_warps']}",
    "- MMA count/barriers/shared layout unchanged: PASS", "", "## Diagnostic GEMM microbenchmark", "",
    "| shape | grid64 median us | L2-grouped median us | delta |", "|---|---:|---:|---:|",
]
for label, row in manifest["diagnostic"].items():
    report.append(f"| {label} | {row['grid_median_us']:.1f} | {row['grouped_median_us']:.1f} | {100*row['median_delta']:+.2f}% |")
report += ["", "## Production prefill", "", "| arm | median session P50 tok/s | median session mean | median P95 latency ms | decode P50 | oracle prefixes | sessions |", "|---|---:|---:|---:|---:|---|---:|"]
for row in manifest["production_summary"]:
    report.append(f"| {row['arm']} | {row['session_p50_median']:.1f} | {row['session_mean_median']:.1f} | {row['session_p95_latency_median_ms']:.2f} | {row['decode_p50_median']:.1f} | {', '.join(row['oracle_prefixes'])} | {row['sessions']} |")
if manifest.get("target_analysis"):
    a = manifest["target_analysis"]
    report += ["", "## 15k feasibility budget", "", f"- measured/target: {a['measured_tps']:.1f} / {a['target_tps']:.0f} tok/s", f"- required speedup: {a['required_speedup']:.2f}x", f"- measured/target prompt time: {a['measured_prefill_ms']:.2f} / {a['target_prefill_ms']:.2f} ms", f"- diagnostic attention/GEMM stage share: {100*a['attention_share']:.1f}% / {100*a['gemm_share']:.1f}%", f"- diagnostic infinite-attention-only ceiling: {a['infinite_attention_ceiling_tps']:.1f} tok/s", f"- diagnostic infinite-GEMM-only ceiling: {a['infinite_gemm_ceiling_tps']:.1f} tok/s"]
d = manifest.get("wave9_decision")
report += ["", "## Wave 9 gate", ""]
if d:
    report += [f"- median P50 delta: {100*d['median_delta']:+.2f}%", f"- paired evidence: {json.dumps(d['paired'])}", f"- prefill P50/mean gates: {d['prefill_p50_pass']} / {d['prefill_mean_pass']}", f"- P95 tail/decode gates: {d['tail_pass']} / {d['decode_pass']}", f"- exact glproc next-token parity: {d['oracle_parity_pass']}", f"- verdict: {'RETAIN' if d['retain'] else 'REJECT/HOLD'}"]
else:
    report.append("- verdict: PENDING")
if manifest.get("ncu", {}).get("available") and not any(x.get("permitted") for x in manifest["ncu"].get("runs", {}).values()):
    report.append("- NCU counters unavailable: ERR_NVGPUCTRPERM; PTXAS/static resource evidence remains archived.")
report += ["", "## Interpretation rule", "", "Retain only if both paired sessions improve prefill P50 and mean by at least 5%, P95 prefill latency and decode do not regress by more than 5%, exact greedy next-token parity and hardware bit-parity stay green, PTXAS reports zero spills, and the candidate remains in the >=24-warp resource tier.", "", "The CTA-ordering microbenchmark and telemetry stage shares are diagnostic only."]
(RESULTS / "WAVE9_REPORT.md").write_text("\n".join(report), encoding="utf-8")
archive = Path(shutil.make_archive(str(WORK / "glcuda_t4_ceiling_wave9_fetch_results"), "zip", root_dir=RESULTS))
print(f"Results directory: {RESULTS}")
print(f"Download archive: {archive}")
print(f"Archive size: {archive.stat().st_size / 1e6:.2f} MB")
print("\n" + "\n".join(report))
